---
toc: false
title: Why Causality — the Prediction-Decision Gap
date: "10/30/2026"
description: Correlation is not causation, except when it is.
categories: causal_inference
image: /enso-thumbnail.jpg
image-alt: The enso circle
draft: true
---


## Why causal inference 

### The Purpose of Data Science

Every once in a while my mom asks me if I can re-explain to her what data scientists do. It's a great question really—we in the field should ask ourselves more often. In a single sentence I'd say: the purpose of data science is to create value from data.  And what's valuable to the organizations we serve?  Well an organization has goals, and in order to achieve those goals it needs to make decisions about which actions to take. At its most effective, data science is about decision intelligence—informing a decision making process to help achieve desired outcomes. These decisions come in many forms, from executive deliberation on strategic bets to streaming automations occurring millions of times per day.

Examples include:
* What price to charge for this ride share trip?
* Which product to recommend to this user?
* Should a school system reduce its class sizes?
* Where to deploy mobile health centers in an underserved rural area?

Behind each of these decisions is a set of what if questions of the form: What would happen if I did this? What would happen if I did that? To inform the decision we can predict answers to these what if questions. 

Often these answers are already valuable, but we can go one step further and recommend the action corresponding to the most favorable outcome. Thus our business problems can often be formulated as optimization problems of the form

$$ \underset{x}{\operatorname{argmax}} f(x) $$

where $x$ indexes the possible actions we can take and $f$ is a "what if" function that predicts the benefit of an action. The above expression says: "find the action that maximizes the predicted benefit."

So that's the setup; the key for optimal decision making is having a "what if" function that reliably maps actions to outcomes.

For data scientists of my generation who spent the early part of their careers getting good at machine learning, this probably  sounds an awful lot like a prediction problem.
We could follow the typical ML playbook , training a model that minimizes out of sample prediction error.
That means it would be the best predictor of  outcomes from actions  and an ideal objective function to use in our optimization task right?

Maybe; maybe not.

Unfortunately there are a few ways this can go right and  many ways this can go wrong. 
Using well-fitting statistical models or accuracy-maxing ML models for decision making can fail, not just slightly, but catastrophically. 
Many of these failure modes arise from errors in reasoning about what causes what and how, i.e. causality.
In this post we'll look at two types of prediction-decision failure modes that can occurr because of mishandling causality.
The first is about the importance of the causal mechanisms behind the correlations we exploit for prediction, and the second is about how these mechanisms can shift under our feet when taking a model from one environment to another.
The key idea is that using statistical or ML models for decision making requires careful causal reasoning, and lucky for you, this is the first post in a forthcoming series on Causal Inference.

So let's kick this series off by seeing some of the scary things that can happen when we naively use predictive models for decision making.

## Prediction-Decision Failure Modes

### Correlation does not imply causation — except when it does.

We've all had it drilled into our heads — 
*[correlation does not imply causation](https://en.wikipedia.org/wiki/Correlation_does_not_imply_causation)* (CDNIC) —
which refers to the inability to deduce a cause-and-effect relationship between two events  solely on the basis of an observed correlation or other statistical association between them.
Maybe this reminds you of the
[spurious correlations](https://www.tylervigen.com/spurious-correlations)
between unrelated quantities like the  popularity  of the surprised Pikachu meme and the number of middle school teachers in Puerto Rico.
The keyword is "spurious", meaning those high correlation coefficients are basically the Type-I errors from dredging correlations across small samples of many pairs of unrelated variables. 
Let's set these finite sample issues aside and consider cases where two variables are actually correlated at a population level.

Consider [Reichenbach's Common Cause Principle](https://plato.stanford.edu/entries/physics-Rpcc/), 
which states that if two events are correlated, there must either be a causal relationship between them or a third event that causes both.
That means that a true correlation between two variables tells us that one causes the other, the other causes the one, or another variable causes both. 
So in a sense, correlation *does* imply causation, but alone it cannot tell us *which* causal relationship exists between the variables.
Let that sink in, this means that no matter how much observed data you collect, it's not possible from the data alone to determine which variables are causes and which are effects.
Furthermore, since predictive models simply leverage correlations and associations among variables from a sample of training data, a predictive model cannot tell you what causes what.

This isn't a problem for prediction, but in decision making, what causes what is absolutely critical.
Per Reichenbach, the observed correlations leveraged by predictive models could arise from several different causal mechanisms, 
and if we have the wrong mechanism, we can end up making terribly misguided decisions.


### Wrong Causal Mechanism


The generalized essence is that association is not the effect of an action. A feature can predict an outcome because it causes it, because the outcome causes it, because both share a common cause, or because of selection. Prediction treats all four the same. A decision treats them very differently.

Suppose we're looking at some app useage data, and we want to figure out whether notifications can help to increase user engagement. 

* X: user enables push notifications during their first two weeks.
* Y: engagement, measured by the number of sessions in the first two weeks.

The data shows  a strong positive correlation between notifications and sessions in the data. 
We build a predictive model on this data, which picks up that correlation, predicting higher engagement when users have notifications enabled.
The decision: should we prompt every new user on day 1 to turn notifications on?

There are many different causal mechanisms that could lead to correlation between notifications and engagement.
Whether it's a good idea to nudge users to turn on notifications depends on which mechanism actually governs the system.

### 1. X causes Y

Notifications remind people to come back, so each enabled user opens the app more.

Structure: Notifications → Sessions
If you prompt everyone: Sessions go up. 

### 2. Y causes X (reverse causation)

Suppose the OS notification permission prompt is shown only after a user’s 5th session. 
Users with many sessions see the prompt and many of them accept, while users who open the app only a few times never see it. Notifications themselves do nothing for engagement.

Structure: Sessions → Sees prompt → Notifications on
Tell in the data: Almost nobody with fewer than 5 sessions has notifications on.

If you prompt everyone on day 1: You flip X without touching Y, and engagement doesn’t change.


### 3. Common cause (confounding)

Some users arrive through a friend’s invite (C). They turn notifications on to see what their friends are doing, and they also use the app more because of that social pull. The channel drives both, and notifications don’t affect engagement.

Structure: Notifications ← Invited by friend → Sessions
Tell in the data: Within each acquisition channel, notification-on and notification-off users look the same.

If you prompt everyone: Non-invited users enable notifications and nothing changes for them. Adjusting for channel would have revealed this.



## Failure to Generalize Across Environments

## Looking Ahead